# JN2 · The address key

A "place" shows up differently in every source — `2650 Telegraph Ave` in one file, `2650 Telegraph` in another, `2650 TELEGRAPH AVE` in a third. To a computer those are three different strings, so a naive join treats them as three different places — and a building can silently fall through the crack between two sources.

This notebook builds the fix: an **address key** that decides when two written addresses mean the *same place*. We don't invent one — we import the **real** key the pipeline runs on (`s0_keys`), and meet the trap that makes plain string-equality the wrong tool for the job.

> Clonable + **read-only** — it reads the shared module, never writes.

### Running the cells

To run a cell, click it and press **Shift + Return**, or click the **run (▸) button** on the cell. The simplest way through any notebook here is to start at the top and run each cell in order, reading the output that appears beneath it.

Some of the computational cells may look complex right now — that's expected, and it's fine. **You don't need to understand every line yet;** the ideas become clear as you go. Run them, watch what they produce, and keep moving.

💡 Tip: the **Next** link opens the following notebook in a new tab. If Colab says you have too many sessions, just close the previous tab and continue.

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN1 · Getting the data in the door](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN1_ingest.ipynb)  |  Next: [JN3 · Build the spine + units](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN3_spine_units.ipynb) →

## (run first) Colab setup

Fetches the data + shared modules from R2. **No-op if you already have the repo locally** (it detects a checkout and skips). On Colab / a bare session it recreates the minimal repo layout under the working directory so the config cell below finds everything unchanged.

In [1]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
_have_repo = (_here/'scripts'/'build_v2').exists() or any((p/'scripts'/'build_v2').exists() for p in _here.parents)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')


local repo detected - no fetch needed


In [2]:
def md(t):
    from IPython.display import Markdown, display
    display(Markdown(t))

## Config — point this at your data

Find the repo root, point at the permit feed, and put the project's real shared code on the path. The two knobs near the top are all a student changes to run another city.

In [3]:
# === CONFIG — point this at YOUR city's permit data (this notebook is clonable) ===
from pathlib import Path
import sys, glob

# walk up to the repo root (where scripts/build_v2 lives) so the notebook runs from anywhere
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# --- the two knobs a student changes for another city ---
PERMIT_GLOB   = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
HEADER_ROW    = 7        # 0-indexed: Berkeley's CPRA export puts the column names on row 8
EXPECTED_UNIQUE = 30764  # the known unique-permit total for YOUR feed (Berkeley = 30,764)

# import the REAL shared modules the pipeline uses (we demonstrate them, never reinvent)
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root :', REPO_ROOT)
print('feed files:', [Path(f).name for f in glob.glob(PERMIT_GLOB)])


repo root : /Users/johngage/berkeley-data
feed files: ['BP_Annual Permit Report-2023-2025.xlsx', 'BP_Annual Permit Report-2018-2022.xlsx']


## First, turn an address into a key

Before we can decide whether two addresses *match*, we have to stop treating them as raw text. `s0_keys.normalize_address` parses a written address into a structured **key** — a house number, a street name, and an *optional* street type — and it's the **same** function the whole pipeline uses.

**Our plan:** import the real function (never a teaching toy that drifts from production) and run it on a few addresses to see what a "key" actually holds.

In [4]:
from s0_keys import normalize_address   # the REAL pipeline key — imported, not reinvented

for a in ['2352 SHATTUCK Ave', '2650 Telegraph Ave', '2650 Telegraph']:   # same place, different surface text
    k = normalize_address(a)            # parse the string into a structured key
    print(f'{a!r:22} -> {k!r:22}  number={k.number}  street={k.street!r}  stype={k.stype!r}  bucket={k.bucket}')

'2352 SHATTUCK Ave'    -> <2352 SHATTUCK AVE>     number=2352  street='SHATTUCK'  stype='AVE'  bucket=('2352', 'SHATTUCK')
'2650 Telegraph Ave'   -> <2650 TELEGRAPH AVE>    number=2650  street='TELEGRAPH'  stype='AVE'  bucket=('2650', 'TELEGRAPH')
'2650 Telegraph'       -> <2650 TELEGRAPH>        number=2650  street='TELEGRAPH'  stype=None  bucket=('2650', 'TELEGRAPH')


In [5]:
_t1 = normalize_address('2650 Telegraph Ave')   # written with a type
_t2 = normalize_address('2650 Telegraph')       # written without one
md(f'''## What just happened

`normalize_address` turned each string into a structured **key**: a number, a street, and an *optional* type. The two Telegraph addresses land on the **same bucket** `{_t2.bucket}` — yet one carries `stype={_t1.stype!r}` and the other `stype={_t2.stype!r}`. Same place, different surface text. That gap — identical bucket, different type — is exactly what the next step has to reconcile. And note we used the pipeline's own function, not a lookalike.''')

## What just happened

`normalize_address` turned each string into a structured **key**: a number, a street, and an *optional* type. The two Telegraph addresses land on the **same bucket** `('2650', 'TELEGRAPH')` — yet one carries `stype='AVE'` and the other `stype=None`. Same place, different surface text. That gap — identical bucket, different type — is exactly what the next step has to reconcile. And note we used the pipeline's own function, not a lookalike.

In [6]:
import pandas as pd
# a focused look: different surface strings, and the key each one collapses to
_examples = ['2352 SHATTUCK Ave', '2650 Telegraph Ave', '2650 Telegraph', '2650 Telegraph Way']
pd.DataFrame([{'written': s,
               'number': normalize_address(s).number,
               'street': normalize_address(s).street,
               'type':   normalize_address(s).stype,
               'bucket': normalize_address(s).bucket} for s in _examples])

,written,number,street,type,bucket
0,2352 SHATTUCK Ave,2352,SHATTUCK,AVE,"(2352, SHATTUCK)"
1,2650 Telegraph Ave,2650,TELEGRAPH,AVE,"(2650, TELEGRAPH)"
2,2650 Telegraph,2650,TELEGRAPH,None,"(2650, TELEGRAPH)"
3,2650 Telegraph Way,2650,TELEGRAPH,WAY,"(2650, TELEGRAPH)"


## Trap-lesson: the same place, written differently — you need a RELATION, not `==`

Look at the two Telegraph rows above: `<2650 TELEGRAPH AVE>` and `<2650 TELEGRAPH>`. As **strings**
they are not equal, so a naive `==` join drops the match — and the building silently looks like it
exists in one source but not the other. (This exact bug, in the real comparison, falsely flagged
**357 buildings** as missing until the relation was used.)

The fix is not to *store* one canonical string (you can't make one string equal both "Ave" and bare).
The fix is a **relation**: `AddressKey.matches()` — same number + street, and type-compatible
(equal types, **or either type absent** = wildcard). Two *different present* types do **not** match.

**Our plan:** take a pair we *know* is the same place, watch plain `==` drop it, then watch `matches()` get it right.

In [7]:
a = normalize_address('2650 Telegraph Ave')
b = normalize_address('2650 Telegraph')        # suffix absent
c = normalize_address('2650 Telegraph Way')    # a DIFFERENT present type

# the same place, three comparisons: wildcard relation, different-type relation, naive string ==
print('Ave  vs (no type) :', a.matches(b), ' <- wildcard: absent type matches present')
print('Ave  vs Way       :', a.matches(c), ' <- two different present types never match')
print('string == (naive) :', repr(a) == repr(b), ' <- the bug a relation avoids')

Ave  vs (no type) : True  <- wildcard: absent type matches present
Ave  vs Way       : False  <- two different present types never match
string == (naive) : False  <- the bug a relation avoids


In [8]:
_p = normalize_address('2650 Telegraph Ave')
_q = normalize_address('2650 Telegraph')
_r = normalize_address('2650 Telegraph Way')
md(f'''## What just happened

A **relation** reads meaning where `==` only reads characters. `Ave` vs *no type* → **{_p.matches(_q)}** (the absent type acts as a wildcard); `Ave` vs `Way` → **{_p.matches(_r)}** (two *different present* types never match); and plain `==` on the strings → **{repr(_p)==repr(_q)}** — the silent miss a relation avoids. The join now matches *places*, not spellings.''')

## What just happened

A **relation** reads meaning where `==` only reads characters. `Ave` vs *no type* → **True** (the absent type acts as a wildcard); `Ave` vs `Way` → **False** (two *different present* types never match); and plain `==` on the strings → **False** — the silent miss a relation avoids. The join now matches *places*, not spellings.

In [9]:
import pandas as pd
# where the relation and the naive string check disagree
_pairs = [('2650 Telegraph Ave', '2650 Telegraph'),
          ('2650 Telegraph Ave', '2650 Telegraph Way'),
          ('2650 Telegraph Ave', '2650 Telegraph Ave')]
pd.DataFrame([{'left': L, 'right': R,
               'matches() relation': normalize_address(L).matches(normalize_address(R)),
               'naive string ==':    repr(normalize_address(L)) == repr(normalize_address(R))}
              for L, R in _pairs])

,left,right,matches() relation,naive string ==
0,2650 Telegraph Ave,2650 Telegraph,True,False
1,2650 Telegraph Ave,2650 Telegraph Way,False,False
2,2650 Telegraph Ave,2650 Telegraph Ave,True,True


## The bucket index + the ambiguity guard

`matches()` is a pairwise relation; for speed you index candidates by `bucket = (number, street)`
(type-agnostic) and apply `matches()` within the bucket. But a wildcard has a danger: if a bucket
holds **two** present types (`Ave` *and* `Way`), an absent-type key matches **both** — that is
**ambiguous**, and the rule is *do not auto-match* (flag it). This is the guard that keeps the
wildcard safe.

**Our plan:** put two present types in one bucket, ask a type-less key to match, and confirm the guard refuses to guess.

In [10]:
# index by bucket (number, street), then apply matches() within the bucket
absent = normalize_address('2650 Telegraph')                 # no street type
candidates = [normalize_address('2650 Telegraph Ave'),
              normalize_address('2650 Telegraph Way')]         # bucket holds TWO present types
matched_types = {c.stype for c in candidates if absent.matches(c)}
ambiguous = absent.stype is None and len(matched_types) > 1
print('absent-type matches present types:', matched_types)
print('ambiguity guard fires (do NOT auto-match):', ambiguous)

absent-type matches present types: {'AVE', 'WAY'}
ambiguity guard fires (do NOT auto-match): True


In [11]:
_ab = normalize_address('2650 Telegraph')
_cands = [normalize_address('2650 Telegraph Ave'), normalize_address('2650 Telegraph Way')]
_mt = {c.stype for c in _cands if _ab.matches(c)}
_fires = _ab.stype is None and len(_mt) > 1
md(f'''## What just happened

Bucketing by `(number, street)` makes matching fast — but the wildcard needs a brake. Here a type-less key matches **{len(_mt)}** present types ({_mt}), so the ambiguity guard fires (**{_fires}**) and we *refuse to auto-match* rather than guess wrong. A safe wildcard is one that knows when to stop.''')

## What just happened

Bucketing by `(number, street)` makes matching fast — but the wildcard needs a brake. Here a type-less key matches **2** present types ({'AVE', 'WAY'}), so the ambiguity guard fires (**True**) and we *refuse to auto-match* rather than guess wrong. A safe wildcard is one that knows when to stop.

## The checkpoint: verify before you trust

We've defined the key and the relation — but how do we *know* they behave? Three cheap assertions pin the contract down: the wildcard match holds, two different present types stay apart, and the ambiguity guard fires. If someone later "simplifies" the match logic, one of these breaks loudly — now, not three notebooks downstream.

In [12]:
# 1) suffix-present <-> suffix-absent matches
assert normalize_address('2650 Telegraph Ave').matches(normalize_address('2650 Telegraph'))
# 2) two different present types do NOT match
assert not normalize_address('2650 Telegraph Ave').matches(normalize_address('2650 Telegraph Way'))
# 3) the ambiguity guard fires on absent -> multiple present
ab = normalize_address('2650 Telegraph')
pm = {c.stype for c in [normalize_address('2650 Telegraph Ave'),
                        normalize_address('2650 Telegraph Way')] if ab.matches(c)}
assert ab.stype is None and len(pm) > 1

print('CHECKPOINT PASS')
print('  suffix pair matches (wildcard) - two-different-present do NOT - ambiguity guard fires')

CHECKPOINT PASS
  suffix pair matches (wildcard) - two-different-present do NOT - ambiguity guard fires


In [13]:
md('''## What just happened

All three assertions passed: the wildcard matched, the two different present types stayed apart, and the guard caught the ambiguous case. That's the course habit in one cell — **state what must be true, then make the computer prove it.** This address key is the quiet foundation JN3 builds the spine on.''')

## What just happened

All three assertions passed: the wildcard matched, the two different present types stayed apart, and the guard caught the ambiguous case. That's the course habit in one cell — **state what must be true, then make the computer prove it.** This address key is the quiet foundation JN3 builds the spine on.

**JN2 done.** You can now relate the same place across sources safely. **Next — JN3:** the spine and
`net_units`, and the trap where one column means different things on different permit types.

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN1 · Getting the data in the door](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN1_ingest.ipynb)  |  Next: [JN3 · Build the spine + units](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN3_spine_units.ipynb) →